# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading, exploring, and processing the FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR<sup>2</sup> dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Obtain the metadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available Record Sets, Fields, and Column `@id`s.

We'll list the dataset's Record Sets and their available fields, referencing each by its Croissant `@id`.

You can use these `@id`s in subsequent queries and extractions.


In [ ]:
# List available Record Sets and their Field @id's
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set name: {rs.name}\n  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields]
    print(f"  Fields @id's:")
    for f in rs.fields:
        print(f"    - {f.id} (name: {f.name})")
    print()


## 3. Data Extraction

Load data from a specific Record Set into a DataFrame for analysis. Use the Record Set and Field `@id`s from the overview above.

For this dataset, there is typically a main Record Set containing the protein table. We'll select the first available Record Set as the main table for demonstration, but you can change `record_set_id` below to work with any of the available Record Sets.


In [ ]:
# Extract data from record sets (@id references)
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    # load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Select the first record set for demonstration (replace with desired @id if needed)
main_record_set_id = record_set_ids[0]

print(f"Columns for record set {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll analyze and process the data:

- Select a numeric field by its `@id` (e.g., protein abundance or coverage_pct)
- Filter records by value
- Normalize values
- Group by a categorical `@id` (e.g., protein description)


> **Note:** Replace the example `numeric_field_id` and `group_field_id` with fields appropriate for your use case by copying from the field list above.


In [ ]:
# Choose numeric and categorical fields by @id for analysis
# (Edit as appropriate for this dataset)

# Demonstrate using 'cr:field:coverage_pct' and 'cr:field:description' as examples.
# Replace these @ids with those listed above as appropriate.

numeric_field_id = None
group_field_id = None

# Try to automatically guess a numeric field and a descriptive group field
df = dataframes[main_record_set_id]
if len(df.columns) > 0:
    # Heuristically pick the first float or int column for numeric field
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Pick a non-numeric column for grouping
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break
else:
    raise RuntimeError("No columns found in main DataFrame.")

print(f"Numeric field chosen for analysis: {numeric_field_id}")
print(f"Categorical field chosen for grouping: {group_field_id}")

# Filter: retain only records where numeric field > threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (first 5 rows):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records (first 5 rows):")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the chosen field and aggregate
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (first 5 rows):")
    print(grouped_df.head())

## 5. Visualization

Plot distributions and relationships between fields in the selected Record Set. For example, histogram of a numeric field and mean per group.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the main numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot grouped by the categorical field
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(12, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id} (filtered)")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We have loaded the Mass Spectrometry EV dataset via its Croissant schema using `mlcroissant`.
- The notebook demonstrated how to discover available Record Sets and Fields (referenced by their Croissant `@id` fields), extract data into DataFrames, and run basic EDA including filtering, normalization, grouping, and visualization with clear provenance.
- For further analysis, use the discovered field and Record Set `@id`s in custom processing according to your research question or workflow.

Explore the FAIR^2 schema for additional details on metadata, context, and associated documentation.
